In [1]:
df_raw = spark.read.option("multiline", "true").json("Files/raw/weather_moscow.json")
df_raw.printSchema()

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 3, Finished, Available, Finished, False)

root
 |-- daily: struct (nullable = true)
 |    |-- precipitation_sum: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- temperature_2m_min: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |    |-- time: array (nullable = true)
 |    |    |-- element: string (containsNull = true)
 |    |-- windspeed_10m_max: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- daily_units: struct (nullable = true)
 |    |-- precipitation_sum: string (nullable = true)
 |    |-- temperature_2m_max: string (nullable = true)
 |    |-- temperature_2m_min: string (nullable = true)
 |    |-- time: string (nullable = true)
 |    |-- windspeed_10m_max: string (nullable = true)
 |-- elevation: double (nullable = true)
 |-- generationtime_ms: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- lo

In [2]:
from pyspark.sql.functions import col, arrays_zip, explode

df_zipped = df_raw.select(
    explode(
        arrays_zip(
            col("daily.time").alias("date"),
            col("daily.temperature_2m_max").alias("temp_max"),
            col("daily.temperature_2m_min").alias("temp_min"),
            col("daily.precipitation_sum").alias("precipitation"),
            col("daily.windspeed_10m_max").alias("windspeed_max")
        )
    ).alias("day")
)

df_zipped.printSchema()
display(df_zipped)

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 6, Finished, Available, Finished, False)

root
 |-- day: struct (nullable = false)
 |    |-- date: string (nullable = true)
 |    |-- temp_max: double (nullable = true)
 |    |-- temp_min: double (nullable = true)
 |    |-- precipitation: double (nullable = true)
 |    |-- windspeed_max: double (nullable = true)



SynapseWidget(Synapse.DataFrame, e41ce92e-c461-4362-9d0a-ff5b04c73b17)

In [3]:
df_weather = df_zipped.select(
    col("day.date").alias("Date"),
    col("day.temp_max").alias("TempMax"),
    col("day.temp_min").alias("TempMin"),
    col("day.precipitation").alias("Precipitation"),
    col("day.windspeed_max").alias("WindSpeedMax")
)

df_weather.printSchema()
display(df_weather)

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 7, Finished, Available, Finished, False)

root
 |-- Date: string (nullable = true)
 |-- TempMax: double (nullable = true)
 |-- TempMin: double (nullable = true)
 |-- Precipitation: double (nullable = true)
 |-- WindSpeedMax: double (nullable = true)



SynapseWidget(Synapse.DataFrame, c2f94a84-2b70-433a-9536-9269353c8a02)

In [4]:
from pyspark.sql.functions import to_date

df_weather = df_weather.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

df_weather.printSchema()

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 8, Finished, Available, Finished, False)

root
 |-- Date: date (nullable = true)
 |-- TempMax: double (nullable = true)
 |-- TempMin: double (nullable = true)
 |-- Precipitation: double (nullable = true)
 |-- WindSpeedMax: double (nullable = true)



In [6]:
from pyspark.sql.functions import min as spark_min, max as spark_max, count

df_weather.select(
    spark_min("Date").alias("min_date"),
    spark_max("Date").alias("max_date"),
    count("Date").alias("row_count")
).show()

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 10, Finished, Available, Finished, False)

+----------+----------+---------+
|  min_date|  max_date|row_count|
+----------+----------+---------+
|2021-01-01|2023-12-31|     1095|
+----------+----------+---------+



In [7]:
df_weather.write.format("delta").mode("overwrite").saveAsTable("fact_weather")

StatementMeta(, 0def3bd7-e89d-4fbc-bf4e-c9b2f81654fa, 11, Finished, Available, Finished, False)

In [1]:
from pyspark.sql.functions import sequence, to_date, explode, col, year, month, quarter, dayofweek, date_format, when

dim_date = spark.sql("""
    SELECT explode(sequence(to_date('2021-01-01'), to_date('2023-12-31'), interval 1 day)) AS FullDate
""")

dim_date = dim_date.withColumn("Year", year("FullDate")) \
    .withColumn("Quarter", quarter("FullDate")) \
    .withColumn("Month", month("FullDate")) \
    .withColumn("MonthName", date_format("FullDate", "MMMM")) \
    .withColumn("DayOfWeek", date_format("FullDate", "EEEE")) \
    .withColumn(
    "Season",
    when(col("Month").isin(12, 1, 2), "Winter")
    .when(col("Month").isin(3, 4, 5), "Spring")
    .when(col("Month").isin(6, 7, 8), "Summer")
    .otherwise("Fall")
)

display(dim_date)

StatementMeta(, 2f5105b5-69b1-4de2-87b1-ef98c48c241b, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5758f01f-90fd-4295-ad8f-8fa7aa0bca12)

In [2]:
dim_date.write.format("delta").mode("overwrite").saveAsTable("dim_date")

StatementMeta(, 2f5105b5-69b1-4de2-87b1-ef98c48c241b, 4, Finished, Available, Finished, False)